# Validation — `kappa-lora-spectral-targeting`

**What this measures:** `num_trainable_params`, exactly as the repo's own MetaMathQA harness writes it into its result JSON, directly measures the mechanism: κ-selection must keep only ceil(56·0.5)=28 of the 56 q/v candidates, so trainable params drop from full-LoRA r=32's 9,175,040 to at most 5,505,024 (the all-q geometric worst case; 14q+14v expectation is exactly 50.0%), while the `test_accuracy` guardrail (≥0.3636 = the published anchor row 0.38362 minus the repo's ±0.02 parity band) holds the "without losing fit" half of the claim. This mirrors how the requester's Colab notebook validates contributions here: run one experiment config from `method_comparison/MetaMathQA/experiments/` through `run.py` and compare the emitted result JSON against the published corpus rows.

**How I read the claim:** The PR adds a κ-LoRA-style knob — condition_number_top_fraction — that pre-ranks target_modules by weight condition number and injects LoRA only into the top fraction; the claim to validate is that top-50% spectral targeting halves trainable parameters without losing fit vs standard LoRA. The benchmark is this repo's established pattern: a new experiments/kappa-lora/llama-3.2-3B-rank32-frac0.5 config run through method_comparison/MetaMathQA/run.py on Llama-3.2-3B (q/v targets, 5000 steps, GSM8K), compared against the published lora--llama-3.2-3B-rank32.json row. Support means: num_trainable_params in [3,670,016, 5,505,024] (40–60% of the 9,175,040 baseline; exactly 4,587,520 if the ranking splits 14q/14v — the printed κ table and q/v composition are the evidence the mechanism fired), with test_accuracy ≥ row − 0.02 over 3 seeds, and time/memory/forgetting as no-regression guardrails. Caveats: under this protocol the candidate pool is 56 heterogeneous modules, so 'halves' holds in expectation but the realized cut is 40–60% by geometry; and the paper's −16.2% time / −4.5% memory deltas are below measurement noise at this scale (~73 MB of 22.8 GB) and are therefore never counted as support.

- ⚠️ The paper ranks 'top 50% of weight matrices' model-wide; this protocol's candidates are only q_proj+v_proj (56 modules, two shapes). 'Halves params' is exact only if the ranking splits 14q/14v (the 50.0% expectation); the realized fraction is 40–60% depending on where Llama-3.2-3B's κ values land — hence the parameter threshold is the geometric bound 5,505,024, not a bare '50% of modules'.
- ⚠️ The −16.2% time and −4.5% memory claims are not detectable at this protocol: adapters on 2 of 7 linear types are a small FLOP slice, and dropping 4,587,520 params saves ~4.6M×16 B (params+grads+AdamW states) ≈ 73 MB against the row's 22.8 GB accelerator peak (≈0.3%). These become no-regression guardrails, not evidence.
- ⚠️ The in-context published row is AdaLoRA, not standard LoRA (its config lists r=8 while its 18,353,664 trainable count corresponds to 28·(6144+4096+2)·r_eff = 286,776·64, i.e. effective r=64 with per-module +r lora_E). The correct comparator is the corpus's protocol-identical lora--llama-3.2-3B-rank32.json row; at compare time substitute its test_accuracy/total_time. The numeric guardrail below uses the only in-context anchor (0.38362).

**Target metric:** `num_trainable_params`

**Repository:** [mayorquinmachines/peft](https://github.com/mayorquinmachines/peft) at commit [`b46d73c33b92`](https://github.com/mayorquinmachines/peft/commit/b46d73c33b92a825c7009c69b48f2ffe7bd80e6f)

**Benchmark:** the repository's own `method_comparison/MetaMathQA/run.py` over `experiments/lora/llama-3.2-3B-rank32-kappa-top50` — not a synthesized stand-in, so the numbers are comparable to what this repository publishes.

**Nothing here has been executed** — there are no outputs and no result is being claimed. Review the measurement, edit the configuration or criteria if it is wrong, then mention `@remyx validate` to run it on Remyx compute — or run the cells top to bottom yourself on a machine with a GPU.

In [ ]:
# Parameters (Remyx passes the commit it measures as `ref`)
variant = "feature"
ref = ""
seed = 0

## 1. Environment

A CUDA GPU is required; the published protocol peaks above 22 GB.

In [ ]:
!nvidia-smi -L
import sys, torch
print(f"python {sys.version.split()[0]} · torch {torch.__version__} · cuda {torch.cuda.is_available()}")

## 2. The code under test

Clone the repository and check out exactly the commit that was validated, then install it in editable mode so the harness imports this checkout. When this notebook runs on Remyx compute the checkout already exists at that commit, and this cell only confirms it.

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/mayorquinmachines/peft"
COMMIT = ref or "b46d73c33b92a825c7009c69b48f2ffe7bd80e6f"

def _sh(*cmd):
    return subprocess.run(cmd, check=True, text=True, capture_output=True).stdout.strip()

def _at_commit():
    try:
        return os.path.isdir(".git") and _sh("git", "rev-parse", "HEAD").startswith(COMMIT)
    except Exception:
        return False

if not _at_commit():
    if not os.path.isdir("repo"):
        _sh("git", "clone", "--quiet", REPO_URL, "repo")
    os.chdir("repo")
    _sh("git", "fetch", "--quiet", "--depth=1", "origin", COMMIT)
    _sh("git", "checkout", "--quiet", COMMIT)
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", "."], check=True)
ROOT = os.getcwd()
print(ROOT)
print(_sh("git", "log", "-1", "--oneline"))

## 3. Credentials

If the benchmark downloads gated models or datasets it needs a Hugging Face token. In Colab, store it as a secret named `HF_TOKEN`; elsewhere set the environment variable.

In [ ]:
import os
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        pass
print("HF_TOKEN set" if os.environ.get("HF_TOKEN") else "HF_TOKEN not set — gated downloads will fail")

## 4. The experiment configuration

The harness runs a method by its configuration directory. This validation points it at `experiments/lora/llama-3.2-3B-rank32-kappa-top50` (relative to `method_comparison/MetaMathQA`).

`method_comparison/MetaMathQA/experiments/lora/llama-3.2-3B-rank32-kappa-top50/adapter_config.json`:

```json
{
  "auto_mapping": null,
  "base_model_name_or_path": "meta-llama/Llama-3.2-3B",
  "condition_number_top_fraction": 0.5,
  "inference_mode": false,
  "lora_alpha": 8,
  "lora_dropout": 0.0,
  "peft_type": "LORA",
  "r": 32,
  "revision": null,
  "target_modules": ["q_proj", "v_proj"],
  "task_type": "CAUSAL_LM"
}
```

In [ ]:
print(open(os.path.join(ROOT, "method_comparison/MetaMathQA/experiments/lora/llama-3.2-3B-rank32-kappa-top50/adapter_config.json")).read())

## 5. Confirm the change under test is what is loaded

The commit printed here must match the one checked out above.

In [ ]:
import importlib
print(_sh("git", "rev-parse", "HEAD"))

## 6. Run the benchmark

`method_comparison/MetaMathQA/run.py` over `experiments/lora/llama-3.2-3B-rank32-kappa-top50` — a directory of experiments runs each in turn; a single experiment runs once.

In [ ]:
os.chdir(os.path.join(ROOT, "method_comparison/MetaMathQA"))
import glob, importlib, runpy, sys, time
RUN_STARTED = time.time()
configs = sorted(glob.glob("experiments/lora/llama-3.2-3B-rank32-kappa-top50/*/")) or ["experiments/lora/llama-3.2-3B-rank32-kappa-top50"]
for cfg in configs:
    print(f"[remyx] {cfg}")
    sys.argv = ["run.py", cfg.rstrip("/")]
    runpy.run_path("run.py", run_name="__main__")

## 7. Read what the benchmark wrote

Results land under `*results/lora--llama-3.2-3B-rank32-kappa-top50*.json` (relative to `method_comparison/MetaMathQA`) — or wherever this harness writes for a non-default checkout; only a document written by the run above counts. The metrics the criteria are judged against are fields of that document.

In [ ]:
import glob, json, os
PATTERNS = ["*results/lora--llama-3.2-3B-rank32-kappa-top50*.json"]
paths = sorted((p for pat in PATTERNS for p in glob.glob(pat)), key=os.path.getmtime)
paths = [p for p in paths if os.path.getmtime(p) >= RUN_STARTED - 1]
if not paths:
    # Some harnesses write elsewhere depending on the checkout (peft uses
    # temporary_results/ off the main branch): any document this run wrote.
    paths = sorted((p for p in glob.glob("**/*.json", recursive=True)
                    if os.path.getmtime(p) >= RUN_STARTED - 1 and "experiments/" not in p),
                   key=os.path.getmtime)
assert paths, "the benchmark wrote no result document"
doc = json.load(open(paths[-1]))
print("result document:", paths[-1])

def find(obj, key):
    """Last value under `key` anywhere in the document ('test accuracy' matches test_accuracy)."""
    hit = None
    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k).replace(" ", "_") == key and isinstance(v, (int, float)):
                hit = v
            found = find(v, key)
            hit = found if found is not None else hit
    elif isinstance(obj, list):
        for item in obj:
            found = find(item, key)
            hit = found if found is not None else hit
    return hit

METRICS = ["num_trainable_params", "test_accuracy", "total_time", "accelerator_memory_max"]
observed = {name: find(doc, name) for name in METRICS}
print(json.dumps(observed, indent=2))

## 8. Against the criteria

Thresholds come from `.remyx/validation.yaml`, so a failing measurement reports rather than crashes. `baseline` is the published row this repository already ships for the comparison method.

In [ ]:
CRITERIA = [
    {
        "metric": "num_trainable_params",
        "direction": "<=",
        "threshold": 5505024,
        "baseline": 9175040
    },
    {
        "metric": "test_accuracy",
        "direction": ">=",
        "threshold": 0.3636,
        "baseline": 0.38362395754359363
    },
    {
        "metric": "total_time",
        "direction": "<=",
        "threshold": 1494,
        "baseline": 1339.9807203459786
    },
    {
        "metric": "accelerator_memory_max",
        "direction": "<=",
        "threshold": 22796042240,
        "baseline": 22796042240.0
    }
]

print(f"{'metric':<28}{'observed':>16}{'baseline':>16}  criterion")
for c in CRITERIA:
    v = observed.get(c["metric"])
    t = c["threshold"]
    ok = None if v is None or t is None else (v <= t if c["direction"] == "<=" else v >= t)
    mark = "?" if ok is None else ("PASS" if ok else "FAIL")
    fmt = lambda x: (f"{x:.6g}" if isinstance(x, float) else str(x))
    print(f"{c['metric']:<28}{fmt(v):>16}{fmt(c['baseline']):>16}  {c['direction']} {fmt(t)}  {mark}")

## 9. Report

One line, machine-readable — what Remyx records as this run's measurement.

In [ ]:
print(json.dumps(observed))

## 10. What the outcome means

- **All rows pass** → the claim holds at this protocol: `num_trainable_params` <= 5505024 with `test_accuracy` >= 0.3636, `total_time` <= 1494, `accelerator_memory_max` <= 22796042240 holding.
- **`num_trainable_params` fails** → the change does not deliver what the claim says at this protocol.
- **A guardrail fails** → the target may be met at the cost of something the claim promised to keep; look at the run log before drawing a conclusion.
- **No result document** → the benchmark did not finish; the run cell above says why.

## Appendix — the criteria file

`.remyx/validation.yaml` as committed:

```yaml
model:
  provider: zai
loop:
  max_iterations: 8
  fix_code: true

benchmarks:
  - name: kappa-lora-spectral-targeting
    suite:
      harness:
        runner: method_comparison/MetaMathQA/run.py
        experiments: experiments/lora/llama-3.2-3B-rank32-kappa-top50
        # run.py writes results/<method>--<exp>.json on main and temporary_results/<method>--<exp>--<timestamp>.json on branches
        results_glob: "method_comparison/MetaMathQA/*results/lora--llama-3.2-3B-rank32-kappa-top50*.json"
        method: lora
        smoke:
          params_path: method_comparison/MetaMathQA/default_training_params.json
          overrides:
            max_steps: 30      # 5000 -> a handful of optimizer steps
            eval_steps: 15     # evaluate twice during the smoke pass
            max_new_tokens: 32 # short GSM8K generation
      scorer: num_trainable_params
      policy:
        guardrail_veto: true
      metrics:
        - name: num_trainable_params
          direction: min
          threshold: 5505024
          # derivation: full LoRA r=32 on q,v = 28*(32*6144 + 32*4096) = 9,175,040; selection keeps ceil(56*0.5)=28 modules;
          # all-v floor 3,670,016 (40.0%), all-q ceiling 5,505,024 (60.0%), 14q+14v expectation 4,587,520 (exactly 50.0%).
          # Threshold is the geometric worst case; vanilla behavior (no targeting) yields 9,175,040 and fails.
          role: target
        - name: test_accuracy
          direction: max
          threshold: 0.3636
          # 0.38362 (in-context published anchor row) minus the repo's +/-0.02 accuracy parity band (Supra verification notebook convention)
          role: guardrail
        - name: total_time
          direction: min
          threshold: 1494
          # 1339.98 s published row total_time * 1.1149 (notebook's +/-3 min on 26.1 min wall-clock tolerance); cost-only, never gates on the paper's -16.2% claim
          role: cost
        - name: accelerator_memory_max
          direction: min
          threshold: 22796042240
          # published row peak; dropping ~4.6M params saves ~73 MB of optimizer state (~0.3%) — no-regression ceiling only
          role: cost
      baseline:
        source: method_comparison/MetaMathQA/results/lora--llama-3.2-3B-rank32.json
        values:
          num_trainable_params: 9175040
          test_accuracy: 0.38362395754359363
          total_time: 1339.9807203459786
          accelerator_memory_max: 22796042240.0
        note: "num_trainable_params=9,175,040 is geometrically forced for standard LoRA r=32 on q_proj+v_proj (28 layers, bias=none) and equals the lora--llama-3.2-3B-rank32 row; the accuracy/time/memory anchors are read from the in-context adalora--llama-3.2-3B-rank32 row (0.38362 / 1339.98 s / 22.80 GB) and should be substituted with the protocol-identical lora row's values at compare time."

compute:
  tier: gpu
  # published row: 5000 steps ~1098 s train + ~242 s eval = 1340 s total on A100; add gated ~6.5 GB Llama-3.2-3B download, dataset fetch, and queue slack
  timeout_s: 5400

held_constant:
  - "same base model meta-llama/Llama-3.2-3B (28 layers, hidden 3072) and tokenizer across arms"
  - "same harness defaults: 5000 MetaMathQA training steps, GSM8K test evaluation, no training_params.json override"
  - "LoRA protocol keys at the sibling rank32 family: r=32, lora_alpha=8, lora_dropout=0.0, target_modules=[q_proj, v_proj], bias=none"
  - "selection computed once on the pre-training weights at injection (checkpoint-deterministic, not serialized)"
  - "same result-JSON metric names and format as the published corpus rows"

avoid:
  - "do not add a training_params.json to the experiment dir — the harness default IS the published protocol and an override breaks comparability"
  - "do not gate on the paper's -16.2% time / -4.5% memory claims — adapters sit on 2 of 7 linear types, a small FLOP slice; time and memory are cost-only no-regression checks"
  - "no unpinned base-model revision drift between the smoke pass and the real run"
  - "no wall-clock comparison against published rows measured on a different GPU class than the A100 they used"
  - "never treat the training-dynamics kappa analysis (condition numbers decreasing) as a pass/fail metric — run.py does not emit it"

provenance:
  num_trainable_params: "user_guidance"
  test_accuracy: "user_guidance"
  total_time: "claim_analysis:paper_metrics"
  accelerator_memory_max: "claim_analysis:paper_metrics"
  suite: "repo_runner:method_comparison/MetaMathQA/run.py"
  experiments: "user_resource:https://colab.research.google.com/drive/1z73-jtAGrq4HkjvorjFcZ77uMwdmWs56?usp=sharing"
  baseline: "published corpus row adalora--llama-3.2-3B-rank32.json (in context); standard-LoRA comparator lora--llama-3.2-3B-rank32.json per claim_analysis"
  held_constant: "protocol_doc:method_comparison/MetaMathQA/README.md"
  smoke: "inferred: override key names inferred from the notebook's description (5000 steps, periodic valid-accuracy eval, GSM8K generation)"
```